# 03 · Spotting an ARP storm

**Data:** `arp-storm.pcap` (Wireshark sample captures).

A host sending ARP requests in bursts is either scanning the subnet or misbehaving (bad NIC driver, loop, mis-set mask). Both are worth a ticket. The measure here is the peak number of ARP requests per sender in any one-second window.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # nbmake runs notebooks from their own folder
import fetch_samples
import pcaplib

In [ ]:
path = fetch_samples.ensure("arp-storm.pcap")
counts = pcaplib.protocol_counts(path)
print(counts)
assert counts == {"ARP": 622}

In [ ]:
rates = pcaplib.arp_request_rates(path, window=1.0)
for mac, peak in sorted(rates.items(), key=lambda kv: -kv[1]):
    print(f"{mac}  peak {peak} requests/s")

## Threshold
A workstation normally sends a handful of ARP requests per minute. Twenty per second from one sender is a reasonable alert threshold for a small office VLAN; tune it per network.

In [ ]:
suspects = pcaplib.arp_storm_suspects(path, threshold=20)
print(suspects)
assert suspects == [("00:07:0d:af:f4:54", 34)]

**Reading:** one MAC address peaks at 34 ARP requests per second. Next steps from the runbooks: find the switch port for that MAC, check whether the host is an approved scanner, and if not, isolate the port (see `runbooks/`).